# 02 — Preprocessing and Data Splits

## Learning goals

- Explain why we need training, validation, and test sets.
- Create a reproducible validation split from the official training data.
- Normalize image pixels consistently.
- Build efficient TensorFlow input pipelines.

**Data-leakage rule:** the test set is held back until final evaluation. Do not use it to select epochs, tune hyperparameters, or compare repeated experiments.


In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import cifar10

SEED = 42
BATCH_SIZE = 64

(x_train_all, y_train_all), (x_test_raw, y_test_raw) = cifar10.load_data()
y_train_all = y_train_all.squeeze()
y_test = y_test_raw.squeeze()

print("Official training data:", x_train_all.shape)
print("Official test data:", x_test_raw.shape)


2026-11-01 17:51:04.366815: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-11-01 17:51:04.393514: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1793555464.408862   11219 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1793555464.413335   11219 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1793555464.434600   11219 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

Official training data: (50000, 32, 32, 3)
Official test data: (10000, 32, 32, 3)


## 1. Make a reproducible train/validation split

We will reserve 5,000 examples from the 50,000-image training set for validation. The official 10,000-image test set remains untouched until final evaluation.

A fixed random seed makes the split repeatable.


In [2]:
rng = np.random.default_rng(SEED)
indices = rng.permutation(len(x_train_all))

val_size = 5000
val_idx = indices[:val_size]
train_idx = indices[val_size:]

x_train = x_train_all[train_idx]
y_train = y_train_all[train_idx]
x_val = x_train_all[val_idx]
y_val = y_train_all[val_idx]

print("Train:", x_train.shape, y_train.shape)
print("Validation:", x_val.shape, y_val.shape)
print("Test:", x_test_raw.shape, y_test.shape)

assert set(train_idx).isdisjoint(set(val_idx))
assert len(train_idx) + len(val_idx) == len(x_train_all)


Train: (45000, 32, 32, 3) (45000,)
Validation: (5000, 32, 32, 3) (5000,)
Test: (10000, 32, 32, 3) (10000,)


## 2. Normalize the images

CIFAR-10 pixels are integer values in the range 0–255. Dividing by 255 converts them to floating-point values in the range 0–1.

Apply the same transformation to training, validation, and test images.


In [3]:
x_train = x_train.astype("float32") / 255.0
x_val = x_val.astype("float32") / 255.0
x_test = x_test_raw.astype("float32") / 255.0

print("Train range:", float(x_train.min()), "to", float(x_train.max()))
print("Validation range:", float(x_val.min()), "to", float(x_val.max()))
print("Test range:", float(x_test.min()), "to", float(x_test.max()))


Train range: 0.0 to 1.0
Validation range: 0.0 to 1.0
Test range: 0.0 to 1.0


## 3. Build efficient input pipelines

`tf.data.Dataset` provides batching and prefetching. Prefetching allows the input pipeline to prepare batches while the model processes the current batch.

We keep the normalized arrays in memory because CIFAR-10 is small enough for this teaching exercise.


In [4]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(len(x_train), seed=SEED, reshuffle_each_iteration=True)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((x_val, y_val))
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_ds = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

images, labels = next(iter(train_ds))
print("Batch images:", images.shape)
print("Batch labels:", labels.shape)


I0000 00:00:1793555471.916256   11219 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3617 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 6GB Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


Batch images: (64, 32, 32, 3)
Batch labels: (64,)


## Exercises

1. Why do we need a validation set if we already have a test set?
2. What could happen if we normalize training images but not test images?
3. Why is a fixed seed useful for comparing experiments?
4. Why should we avoid tuning a model repeatedly against the test set?

**Next:** build a small CNN using these preprocessing choices.
